# 🎓 대학생 AI 과제·시험 관리 도우미 — 5가지 추가 과제

**학습 목표**: Python으로 과제를 분석하고, 하루 **3시간 / 2시간** 시나리오를 비교한 뒤, 선택적으로 기존 **Qwen3-8B AI Agent**에 짧은 조언을 요청합니다.

- **기본 실습(1~4단계)**: RunPod의 Python만 있으면 실행 가능. **GPU와 FastAPI 불필요**.
- **선택 실습(AI 조언)**: RunPod GPU와 기존 FastAPI 서버(`127.0.0.1:8000`)가 실행 중이어야 함.
- **제출(5단계)**: CSV와 학습계획 TXT. 원본 `05_agent_web_mvp` 파일은 변경하지 않습니다.
- **개인정보**: 학번·실명 등 민감한 정보 대신 과목명과 과제 유형만 입력하세요. 공동 GPU 환경의 공용 폴더에 개인정보를 저장하지 마세요.

> **중요**: AI가 JSON 파싱에 실패해도 **Python으로 작성한 학습 계획과 CSV는 정상 생성**됩니다. 코드 셀은 위에서 아래로 순서대로 실행하세요.

## 1단계 — 과목 4개와 과제 마감일 입력하기

`assignments`의 **과목, 과제명, 마감일, 예상 작업시간, 진행률**을 자신의 일정에 맞게 고칩니다.

- `hours`: 과제를 **처음부터 완료하기까지** 예상되는 총 시간
- `progress`: 현재 진행률(0~100)
- `due`: 마감일(`YYYY-MM-DD`) 또는 `TODAY + timedelta(days=n)`

예시 데이터는 학습용 가상 데이터입니다. 날짜는 실행하는 **한국 날짜**를 기준으로 생성됩니다.

In [1]:
from datetime import datetime, timedelta, date
from zoneinfo import ZoneInfo
from pathlib import Path
import csv
import uuid

TODAY = datetime.now(ZoneInfo("Asia/Seoul")).date()

# ✅ 이 부분을 학생 본인의 실제 과목 4개로 수정하세요.
assignments = [
    {"course": "인공지능", "task": "AI Agent 프로젝트 보고서",
     "due": TODAY + timedelta(days=2), "hours": 7, "progress": 20},
    {"course": "자료구조", "task": "알고리즘 구현 과제",
     "due": TODAY + timedelta(days=5), "hours": 8, "progress": 50},
    {"course": "데이터베이스", "task": "SQL 실습 보고서",
     "due": TODAY + timedelta(days=1), "hours": 3, "progress": 0},
    {"course": "컴퓨터네트워크", "task": "기말시험 대비",
     "due": TODAY + timedelta(days=10), "hours": 12, "progress": 25},
]

# 마감일을 문자열로 입력한 경우에도 처리할 수 있도록 표준화
for item in assignments:
    if isinstance(item["due"], str):
        item["due"] = date.fromisoformat(item["due"])
    assert 0 <= item["progress"] <= 100, "진행률은 0~100 사이여야 합니다."
    assert item["hours"] >= 0, "총 예상 작업시간은 0 이상이어야 합니다."

assert len(assignments) >= 4, "과목/과제 정보를 최소 4개 입력하세요."

print("📅 오늘(한국 기준):", TODAY)
for item in assignments:
    print(f"📘 {item['course']} | {item['task']} | 마감 {item['due']} | "
          f"총 {item['hours']}시간 | 진행률 {item['progress']}%")

📅 오늘(한국 기준): 2026-11-26
📘 인공지능 | AI Agent 프로젝트 보고서 | 마감 2026-11-28 | 총 7시간 | 진행률 20%
📘 자료구조 | 알고리즘 구현 과제 | 마감 2026-12-01 | 총 8시간 | 진행률 50%
📘 데이터베이스 | SQL 실습 보고서 | 마감 2026-11-27 | 총 3시간 | 진행률 0%
📘 컴퓨터네트워크 | 기말시험 대비 | 마감 2026-12-06 | 총 12시간 | 진행률 25%


## 2단계 — 진행률 변경 전·후 남은 작업시간 비교하기

**공식**: `남은 작업시간 = 총 예상 작업시간 × (1 - 진행률 / 100)`

예시에서는 인공지능 과제 진행률을 **20% → 40%**로 변경합니다. 자신의 데이터로 실습할 때는 해당 값을 바꾸어 보세요.

In [2]:
def remaining_hours(item):
    """진행률을 반영한 남은 작업시간을 반환합니다."""
    return round(item["hours"] * (1 - item["progress"] / 100), 2)

# 원본을 유지하면서 복사본을 만들어 진행률 변경을 실험합니다.
updated_assignments = [dict(item) for item in assignments]

# TODO: 다른 과제를 선택하거나, 40을 원하는 진행률로 바꿔 보세요.
updated_assignments[0]["progress"] = 40

progress_comparison = []
for before, after in zip(assignments, updated_assignments):
    before_hours = remaining_hours(before)
    after_hours = remaining_hours(after)
    progress_comparison.append({
        "과목": after["course"],
        "과제": after["task"],
        "변경전_진행률": before["progress"],
        "변경후_진행률": after["progress"],
        "변경전_잔여시간": before_hours,
        "변경후_잔여시간": after_hours,
        "감소한_시간": round(before_hours - after_hours, 2)
    })

print("📊 진행률 변경 전·후 비교")
for row in progress_comparison:
    print(f"- {row['과목']}: {row['변경전_진행률']}%→{row['변경후_진행률']}%, "
          f"남은 시간 {row['변경전_잔여시간']}→{row['변경후_잔여시간']}시간")

# 이후 단계는 변경된 진행률을 적용한 데이터를 사용합니다.
active_assignments = updated_assignments

📊 진행률 변경 전·후 비교
- 인공지능: 20%→40%, 남은 시간 5.6→4.2시간
- 자료구조: 50%→50%, 남은 시간 4.0→4.0시간
- 데이터베이스: 0%→0%, 남은 시간 3.0→3.0시간
- 컴퓨터네트워크: 25%→25%, 남은 시간 9.0→9.0시간


## 3단계 — 하루 3시간 / 2시간 학습계획 비교하기

Python이 **마감일이 빠른 과제부터(EDF)** 하루 최대 작업시간을 배분합니다. 마감일이 지난 과제에 새 시간을 배정하지 않습니다.

**주의**: 이는 하루 총 학습시간만 고려한 **간단한 일정 시뮬레이션**입니다. 수업·시험·휴식·과제 간 선행 조건은 반영하지 않습니다.

In [3]:
def simulate_schedule(items, daily_limit, days=7):
    """마감이 빠른 작업부터 오늘 포함 days일간 작업시간을 배분합니다."""
    if daily_limit <= 0:
        raise ValueError("하루 학습 가능 시간은 0보다 커야 합니다.")

    tasks = [dict(item, remaining=remaining_hours(item)) for item in items]
    tasks.sort(key=lambda x: (x["due"], x["course"]))
    schedule = []

    for day_offset in range(days):
        study_day = TODAY + timedelta(days=day_offset)
        capacity = float(daily_limit)

        for task in tasks:
            if capacity <= 1e-8:
                break
            if task["remaining"] <= 1e-8 or task["due"] < study_day:
                continue

            planned = round(min(capacity, task["remaining"]), 2)
            task["remaining"] = round(task["remaining"] - planned, 2)
            capacity = round(capacity - planned, 2)
            schedule.append({
                "날짜": study_day.isoformat(),
                "과목": task["course"],
                "과제": task["task"],
                "작업시간": planned,
                "마감일": task["due"].isoformat()
            })

    last_day = TODAY + timedelta(days=days - 1)
    # 이번 7일 이내에 마감하지만 작업량이 남으면 기한 준수 위험입니다.
    risk = [task for task in tasks
            if task["due"] <= last_day and task["remaining"] > 1e-8]
    pending_future = [task for task in tasks
                      if task["due"] > last_day and task["remaining"] > 1e-8]

    return {
        "daily_limit": daily_limit,
        "schedule": schedule,
        "tasks": tasks,
        "risk": risk,
        "pending_future": pending_future,
        "planned_hours": round(sum(x["작업시간"] for x in schedule), 2),
        "unfinished_hours": round(sum(x["remaining"] for x in tasks), 2)
    }

plan_3h = simulate_schedule(active_assignments, daily_limit=3)
plan_2h = simulate_schedule(active_assignments, daily_limit=2)

for plan in [plan_3h, plan_2h]:
    print("\n" + "=" * 55)
    print(f"⏰ 하루 {plan['daily_limit']}시간 학습 시나리오")
    print(f"7일간 배정한 학습시간: {plan['planned_hours']}시간")
    print(f"7일 후 남은 작업시간: {plan['unfinished_hours']}시간")
    print(f"기한 준수 위험 과제: {len(plan['risk'])}개")
    for row in plan["schedule"]:
        print(f"  {row['날짜']} | {row['과목']} | {row['작업시간']}시간")
    for task in plan["risk"]:
        print(f"  ⚠️ 마감 위험: {task['task']} (미완료 {task['remaining']}시간)")
    if plan["pending_future"]:
        print("  💡 7일 이후 마감 과제는 잔여시간이 있어도 마감 실패로 판정하지 않습니다.")


⏰ 하루 3시간 학습 시나리오
7일간 배정한 학습시간: 20.2시간
7일 후 남은 작업시간: 0.0시간
기한 준수 위험 과제: 0개
  2026-11-26 | 데이터베이스 | 3.0시간
  2026-11-27 | 인공지능 | 3.0시간
  2026-11-28 | 인공지능 | 1.2시간
  2026-11-28 | 자료구조 | 1.8시간
  2026-11-29 | 자료구조 | 2.2시간
  2026-11-29 | 컴퓨터네트워크 | 0.8시간
  2026-11-30 | 컴퓨터네트워크 | 3.0시간
  2026-12-01 | 컴퓨터네트워크 | 3.0시간
  2026-12-02 | 컴퓨터네트워크 | 2.2시간

⏰ 하루 2시간 학습 시나리오
7일간 배정한 학습시간: 14.0시간
7일 후 남은 작업시간: 6.2시간
기한 준수 위험 과제: 1개
  2026-11-26 | 데이터베이스 | 2.0시간
  2026-11-27 | 데이터베이스 | 1.0시간
  2026-11-27 | 인공지능 | 1.0시간
  2026-11-28 | 인공지능 | 2.0시간
  2026-11-29 | 자료구조 | 2.0시간
  2026-11-30 | 자료구조 | 2.0시간
  2026-12-01 | 컴퓨터네트워크 | 2.0시간
  2026-12-02 | 컴퓨터네트워크 | 2.0시간
  ⚠️ 마감 위험: AI Agent 프로젝트 보고서 (미완료 1.2시간)
  💡 7일 이후 마감 과제는 잔여시간이 있어도 마감 실패로 판정하지 않습니다.


## 4단계 — AI 조언 비교 및 실현 가능성 검토

**① Python의 계산 결과**는 확정적인 비교 근거입니다. **② AI의 조언**은 참고자료로 활용하고, Python 일정과 대조해서 타당성을 검증하세요.

- `RUN_AI = False` 상태에서는 **서버 없이** 실행됩니다.
- 기존 RunPod의 **FastAPI 8000번 포트와 GPU가 동작할 때만** `RUN_AI = True`로 변경하세요.
- 기존 Agent가 긴 답변의 JSON 생성에 실패할 수 있으므로 **아주 짧은 조언만 요청**합니다.
- AI 호출이 실패해도 **Python 일정표와 제출 파일은 정상 생성**됩니다.

In [4]:
# ✅ FastAPI 서버와 GPU가 준비되었다면 True로 변경합니다.
RUN_AI = False
BASE_URL = "http://127.0.0.1:8000"


def summarize_for_ai(plan):
    """긴 일정표 대신 검증된 숫자 요약만 전달합니다."""
    risky = ", ".join(t["task"] for t in plan["risk"]) or "없음"
    return (f"하루 {plan['daily_limit']}시간, 7일간 배정 "
            f"{plan['planned_hours']}시간, 남은 일 {plan['unfinished_hours']}시간, "
            f"마감 위험: {risky}")


def ask_agent_short(question):
    """기존 /agent를 호출하고 실패 이유를 반환합니다. 서버 파일은 수정하지 않습니다."""
    try:
        import requests
        response = requests.post(
            BASE_URL + "/agent",
            json={"query": question},
            timeout=240
        )
        response.raise_for_status()
        data = response.json()
        if data.get("status") == "success":
            return {"status": "success", "text": data.get("final_answer", "")}
        return {"status": "agent_error", "text": str(data.get("decision"))}
    except Exception as exc:
        return {"status": "connection_error", "text": str(exc)}


if RUN_AI:
    ai_3h = ask_agent_short(
        "너는 학습 코치다. 다음 결과에 대해 한국어 두 문장으로만 조언해줘. "
        + summarize_for_ai(plan_3h)
    )
    ai_2h = ask_agent_short(
        "너는 학습 코치다. 다음 결과에 대해 한국어 두 문장으로만 조언해줘. "
        + summarize_for_ai(plan_2h)
    )
else:
    ai_3h = {"status": "not_run", "text": "AI 호출 안 함 (RUN_AI=False)"}
    ai_2h = {"status": "not_run", "text": "AI 호출 안 함 (RUN_AI=False)"}

print("📌 3시간 시나리오:", summarize_for_ai(plan_3h))
print("🤖 AI 응답 상태:", ai_3h["status"])
print("🤖 AI 조언:", ai_3h["text"])
print()
print("📌 2시간 시나리오:", summarize_for_ai(plan_2h))
print("🤖 AI 응답 상태:", ai_2h["status"])
print("🤖 AI 조언:", ai_2h["text"])

# TODO: AI 제안과 Python 일정표를 비교한 자신의 검토 의견을 적으세요.
student_review = (
    "[학생 작성] 3시간과 2시간 시나리오를 비교하고, "
    "AI 제안의 실현 가능성을 검토하세요. "
    "AI를 호출하지 않았다면 Python 분석 결과를 근거로 작성하세요."
)
print("\n📝 학생 검토:", student_review)

📌 3시간 시나리오: 하루 3시간, 7일간 배정 20.2시간, 남은 일 0.0시간, 마감 위험: 없음
🤖 AI 응답 상태: not_run
🤖 AI 조언: AI 호출 안 함 (RUN_AI=False)

📌 2시간 시나리오: 하루 2시간, 7일간 배정 14.0시간, 남은 일 6.2시간, 마감 위험: AI Agent 프로젝트 보고서
🤖 AI 응답 상태: not_run
🤖 AI 조언: AI 호출 안 함 (RUN_AI=False)

📝 학생 검토: [학생 작성] 3시간과 2시간 시나리오를 비교하고, AI 제안의 실현 가능성을 검토하세요. AI를 호출하지 않았다면 Python 분석 결과를 근거로 작성하세요.


## 5단계 — CSV와 학습계획 결과 저장하기

**제출 파일**: `assignment_summary.csv`, `progress_comparison.csv`, `schedule_3h.csv`, `schedule_2h.csv`, `learning_plan_comparison.txt` 및 이 Notebook(`.ipynb`).

- RunPod에서는 **`/workspace/student_planner_outputs/`** 아래에 실행별 폴더를 만듭니다.
- 기존 프로젝트 소스와 다른 폴더를 사용하여 **파일 덮어쓰기를 방지**합니다.
- `AI 응답 상태: not_run/agent_error`인 경우에도 결과 파일이 만들어집니다. AI 성공 결과와 혼동하지 마세요.

In [5]:
def save_csv(path, records, columns):
    with open(path, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=columns)
        writer.writeheader()
        for record in records:
            writer.writerow({name: record.get(name, "") for name in columns})


base_dir = Path("/workspace") if Path("/workspace").is_dir() else Path.cwd()
run_id = datetime.now().strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6]
output_dir = base_dir / "student_planner_outputs" / run_id
output_dir.mkdir(parents=True, exist_ok=False)

summary_records = [
    {
        "과목": t["course"],
        "과제": t["task"],
        "마감일": t["due"].isoformat(),
        "D-day": (t["due"] - TODAY).days,
        "총예상시간": t["hours"],
        "진행률": t["progress"],
        "남은시간": remaining_hours(t)
    }
    for t in active_assignments
]

save_csv(output_dir / "assignment_summary.csv", summary_records,
         ["과목", "과제", "마감일", "D-day", "총예상시간", "진행률", "남은시간"])

save_csv(output_dir / "progress_comparison.csv", progress_comparison,
         ["과목", "과제", "변경전_진행률", "변경후_진행률",
          "변경전_잔여시간", "변경후_잔여시간", "감소한_시간"])

schedule_columns = ["날짜", "과목", "과제", "작업시간", "마감일"]
save_csv(output_dir / "schedule_3h.csv", plan_3h["schedule"], schedule_columns)
save_csv(output_dir / "schedule_2h.csv", plan_2h["schedule"], schedule_columns)

comparison_text = "\n".join([
    "대학생 AI 과제·시험 관리 도우미: 학습계획 비교",
    f"작성 기준일(한국): {TODAY}",
    "",
    "[하루 3시간 - Python 계산]",
    summarize_for_ai(plan_3h),
    "마감 위험 상세: " + (
        ", ".join(f"{t['task']} ({t['remaining']}시간 미완료)"
                  for t in plan_3h["risk"]) or "없음"),
    "AI 응답 상태: " + ai_3h["status"],
    "AI 조언: " + ai_3h["text"],
    "",
    "[하루 2시간 - Python 계산]",
    summarize_for_ai(plan_2h),
    "마감 위험 상세: " + (
        ", ".join(f"{t['task']} ({t['remaining']}시간 미완료)"
                  for t in plan_2h["risk"]) or "없음"),
    "AI 응답 상태: " + ai_2h["status"],
    "AI 조언: " + ai_2h["text"],
    "",
    "[학생의 AI 결과 검토]",
    student_review,
    "",
    "주의: 일정은 마감일과 하루 총 시간만 반영한 단순 시뮬레이션입니다."
])
(output_dir / "learning_plan_comparison.txt").write_text(
    comparison_text, encoding="utf-8-sig"
)

print("✅ 제출 결과물 저장 완료")
for path in sorted(output_dir.iterdir()):
    print(" -", path.name)
print("📁 저장 위치:", output_dir)
print("📌 현재 노트북(.ipynb)도 별도로 저장해 제출하세요.")

✅ 제출 결과물 저장 완료
 - assignment_summary.csv
 - learning_plan_comparison.txt
 - progress_comparison.csv
 - schedule_2h.csv
 - schedule_3h.csv
📁 저장 위치: /workspace/student_planner_outputs/20261126_072303_1ce674
📌 현재 노트북(.ipynb)도 별도로 저장해 제출하세요.


## 제출 전 자가 점검

- [ ] 실제 과목·과제 정보 **최소 4개**를 입력했는가?
- [ ] 진행률 변경 전·후의 잔여시간이 정상적으로 비교되었는가?
- [ ] 3시간/2시간 학습계획과 마감 위험을 비교했는가?
- [ ] AI의 제안을 무조건 신뢰하지 않고 Python 일정표와 대조했는가?
- [ ] CSV 4개, TXT 1개, Notebook 1개를 제출할 준비가 되었는가?

**심화 도전**: 마감일이 지난 과제를 추가하고, `진행률 100%`인 과제의 남은 시간이 0인지 확인해 보세요.